# Tea Leaf Disease Detection - GPU Training

**Target: 95%+ mAP50 accuracy**

This notebook trains a YOLOv8 model on Google Colab's free GPU.

## Instructions:
1. Go to Runtime > Change runtime type > Select **GPU**
2. Upload your dataset zip file when prompted
3. Run all cells
4. Download the trained model at the end

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi
print("\n" + "="*50)
print("If you see GPU info above, you're ready to go!")
print("If not, go to Runtime > Change runtime type > GPU")
print("="*50)

In [ ]:
# Step 2: Install required packages
!pip install ultralytics -q
print("Ultralytics installed!")

In [ ]:
# Step 3: Upload your dataset
# Option A: Upload from Google Drive
from google.colab import drive
drive.mount('/content/drive')

# After mounting, set your dataset path:
# DATASET_PATH = '/content/drive/MyDrive/your_dataset_folder'

In [ ]:
# Option B: Upload zip file directly
from google.colab import files
import zipfile
import os

print("Upload your dataset zip file (containing train/, valid/, test/ folders and data.yaml)")
uploaded = files.upload()

# Extract the zip file
for filename in uploaded.keys():
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')
    print("Extraction complete!")

# List contents
!ls -la /content/dataset/

In [ ]:
# Step 4: Create/Update data.yaml with correct paths
import yaml

data_config = {
    'train': '/content/dataset/train/images',
    'val': '/content/dataset/valid/images',
    'test': '/content/dataset/test/images',
    'nc': 3,
    'names': ['blister_blight', 'healthy', 'red_rust']
}

with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("data.yaml created:")
!cat /content/dataset/data.yaml

In [ ]:
# Step 5: Verify dataset
import os

train_images = len(os.listdir('/content/dataset/train/images'))
valid_images = len(os.listdir('/content/dataset/valid/images'))
test_images = len(os.listdir('/content/dataset/test/images')) if os.path.exists('/content/dataset/test/images') else 0

print("="*50)
print("DATASET SUMMARY")
print("="*50)
print(f"Training images: {train_images}")
print(f"Validation images: {valid_images}")
print(f"Test images: {test_images}")
print(f"Total: {train_images + valid_images + test_images}")
print("="*50)

In [ ]:
# Step 6: Train the model (GPU accelerated!)
from ultralytics import YOLO
from datetime import datetime

print("="*70)
print("  TEA LEAF DISEASE DETECTION - GPU TRAINING")
print("  Target: 95%+ mAP50")
print("="*70)
print(f"Start time: {datetime.now()}")
print()

# Load YOLOv8s model (better accuracy than nano)
model = YOLO('yolov8s.pt')

# Train with optimized settings
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=150,  # More epochs for better accuracy
    imgsz=640,
    batch=16,  # Larger batch size on GPU
    device=0,  # Use GPU
    workers=4,
    project='runs/detect',
    name='tea_leaf_gpu',
    patience=30,
    save=True,
    cache=True,
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=5,
    
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    
    # Close mosaic near end
    close_mosaic=20,
    
    plots=True,
    val=True
)

print()
print(f"End time: {datetime.now()}")

In [ ]:
# Step 7: Evaluate the trained model
from ultralytics import YOLO

# Load best model
best_model = YOLO('runs/detect/tea_leaf_gpu/weights/best.pt')

# Validate
val_results = best_model.val(data='/content/dataset/data.yaml')

print()
print("="*60)
print("  FINAL RESULTS")
print("="*60)
map50 = val_results.box.map50 * 100
map50_95 = val_results.box.map * 100
precision = val_results.box.mp * 100
recall = val_results.box.mr * 100

status = "TARGET MET!" if map50 >= 95 else f"(target: 95%)"
print(f"  mAP50:     {map50:.1f}%  {status}")
print(f"  mAP50-95:  {map50_95:.1f}%")
print(f"  Precision: {precision:.1f}%")
print(f"  Recall:    {recall:.1f}%")

# Per-class results
print("\n  Per-class mAP50:")
class_names = ['blister_blight', 'healthy', 'red_rust']
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):
        ap = val_results.box.ap50[i] * 100
        print(f"    {name}: {ap:.1f}%")
print("="*60)

In [ ]:
# Step 8: View training results
from IPython.display import Image, display
import os

results_dir = 'runs/detect/tea_leaf_gpu'

# Display training curves
print("Training Results:")
if os.path.exists(f'{results_dir}/results.png'):
    display(Image(filename=f'{results_dir}/results.png', width=800))

# Display confusion matrix
print("\nConfusion Matrix:")
if os.path.exists(f'{results_dir}/confusion_matrix_normalized.png'):
    display(Image(filename=f'{results_dir}/confusion_matrix_normalized.png', width=600))

In [ ]:
# Step 9: Test on sample images
from ultralytics import YOLO
from IPython.display import Image, display
import os

model = YOLO('runs/detect/tea_leaf_gpu/weights/best.pt')

# Test on validation images
test_dir = '/content/dataset/valid/images'
test_images = os.listdir(test_dir)[:5]  # First 5 images

print("Sample Predictions:")
for img_name in test_images:
    img_path = os.path.join(test_dir, img_name)
    results = model.predict(img_path, save=True, conf=0.25)
    
    # Display result
    result_path = f'runs/detect/predict/{img_name}'
    if os.path.exists(result_path):
        print(f"\n{img_name}:")
        display(Image(filename=result_path, width=400))

In [ ]:
# Step 10: Download trained model
from google.colab import files
import shutil

# Create a zip with the best model and results
print("Preparing files for download...")

# Create output directory
os.makedirs('download', exist_ok=True)

# Copy best model
shutil.copy('runs/detect/tea_leaf_gpu/weights/best.pt', 'download/best.pt')

# Copy results
shutil.copy('runs/detect/tea_leaf_gpu/results.csv', 'download/results.csv')
shutil.copy('runs/detect/tea_leaf_gpu/results.png', 'download/results.png')
if os.path.exists('runs/detect/tea_leaf_gpu/confusion_matrix_normalized.png'):
    shutil.copy('runs/detect/tea_leaf_gpu/confusion_matrix_normalized.png', 'download/confusion_matrix.png')

# Create zip
shutil.make_archive('tea_leaf_model', 'zip', 'download')

print("\nDownloading trained model...")
print("Save this to: C:\\Users\\lenovo\\OneDrive\\Desktop\\Disease\\runs\\detect\\tea_leaf_gpu\\weights\\")
files.download('tea_leaf_model.zip')

## After Training

1. Download the `tea_leaf_model.zip` file
2. Extract `best.pt` to your local project:
   ```
   C:\Users\lenovo\OneDrive\Desktop\Disease\runs\detect\tea_leaf_gpu\weights\best.pt
   ```
3. Update `analyze_leaves.py` to use the new model path
4. Run your analysis script with the improved model!

